# BME 574 - Lecture 7 companion
# The variational autoencoder

Last notebook ended with a latent space that was useless for generating, and a
diagnosis: nothing in the loss said where the codes should be. This notebook
writes down a loss that does.

Nothing here is asserted. Every claim is checked in code:

1. **The probability**, in code - expectation by sampling, Bayes on a screening
   test, and KL both ways.
2. **How intractable the integral really is**, measured.
3. **The reparameterisation trick**, with the gradient shown to survive it.
4. **The ELBO is a lower bound** - demonstrated numerically, not just derived.
5. A trained VAE, and the same experiments that failed last lecture.
6. The ten-line regression test every VAE implementation should pass.

**Runtime** about 7 minutes on two CPU cores.

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

torch.manual_seed(0); np.random.seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEV)

transform = transforms.Compose([transforms.ToTensor()])
train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True,
                                       transform=transform)
test_set  = torchvision.datasets.MNIST(root='./data', train=False, download=True,
                                       transform=transform)
X  = (train_set.data[:20000].float() / 255).reshape(-1, 784).to(DEV)
Xt = (test_set.data.float() / 255).reshape(-1, 784).to(DEV)
Yt =  test_set.targets.clone()
print('train', tuple(X.shape), '| test', tuple(Xt.shape))

def show(imgs, n=10, title=''):
    a = imgs.detach().cpu().reshape(-1, 28, 28).numpy()[:n]
    fig, ax = plt.subplots(1, n, figsize=(1.1 * n, 1.4))
    for i in range(n):
        ax[i].imshow(a[i], cmap='gray', vmin=0, vmax=1); ax[i].axis('off')
    if title: fig.suptitle(title, y=1.1, fontsize=11)
    plt.show()

---
## 1. The probability, in code

### Thing one: an expectation is a weighted average

$$ \mathbb{E}_{p(z)}\!\left[f(z)\right] \;=\; \int f(z)\,p(z)\,dz \;\approx\; \frac{1}{K}\sum_{k=1}^{K} f(z_k), \qquad z_k \sim p(z) $$

Say the symbol as a sentence every time: *"the average value of $f$, if $z$ comes
from $p$."* The integral is the definition; the sum is how you actually compute
it. **Every expectation in this notebook is computed by sampling.**

In [ ]:
mu_t, sd_t = 1.5, 2.0
rng = np.random.default_rng(0)
exact = mu_t ** 2 + sd_t ** 2                       # E[z^2] = mu^2 + sigma^2
s = rng.normal(mu_t, sd_t, 4000)
run = np.cumsum(s ** 2) / np.arange(1, 4001)
print('exact E[z^2]      %.3f' % exact)
for k in (10, 100, 1000, 4000):
    print('  %5d samples   %.3f' % (k, run[k - 1]))

fig, ax = plt.subplots(figsize=(5.6, 2.6))
ax.plot(np.arange(1, 4001), run, color='#1F3864', lw=1.1)
ax.axhline(exact, color='#9C2B2B', ls='--', lw=2, label='exact (%.2f)' % exact)
ax.set_xscale('log'); ax.set_xlabel('samples'); ax.set_ylabel('running average')
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()
print('\nNoisy, unbiased, free. A VAE uses ONE sample per image per step --')
print('which sounds absurd, and works, because it averages out over the batch.')

### Thing two: Bayes

$$ p(a,b) = p(a\mid b)\,p(b) = p(b\mid a)\,p(a) \qquad\Longrightarrow\qquad p(b \mid a) = \frac{p(a\mid b)\,p(b)}{p(a)} $$

Divide both sides by $p(a)$. That is the whole derivation.

Rather than trust the formula, simulate a screening test: 1% prevalence, the
test catches 95% of real cases, and is right 90% of the time on healthy people.
**You test positive. What is the chance you have the disease?**

Write your guess down before running the cell.

In [ ]:
PREV, SENS, SPEC, N = 0.01, 0.95, 0.90, 2_000_000
rng = np.random.default_rng(1)
has_it = rng.random(N) < PREV
positive = np.where(has_it, rng.random(N) < SENS, rng.random(N) < (1 - SPEC))
sim = has_it[positive].mean()

post = SENS * PREV / (SENS * PREV + (1 - SPEC) * (1 - PREV))   # Bayes, directly
print('simulated %d people' % N)
print('   %d tested positive, of whom %d actually have it' % (positive.sum(), (has_it & positive).sum()))
print('   P(disease | positive), simulated  %.4f' % sim)
print('   P(disease | positive), Bayes      %.4f' % post)
print('\nThe test is "95%% accurate" and a positive result means a %.0f%% chance.' % (100 * post))
print('The false positives come from a much bigger pool: 10% of 9,900 healthy')
print('people is 990 false alarms against 95 true ones.')

In a VAE the same four words appear:

| name | symbol | what it is here |
|---|---|---|
| prior | $p(z)$ | what a code looks like before you see any image - **we choose this** |
| likelihood | $p_\theta(x \mid z)$ | the decoder: given a code, what image comes out |
| **posterior** | $p_\theta(z \mid x)$ | given THIS image, which codes could have made it |
| evidence | $p_\theta(x)$ | how probable this image is under the whole model |

The posterior is what we want and cannot compute. Everything that follows is
about that.

### Thing three: KL divergence

$$ D_{\mathrm{KL}}(q \,\|\, p) \;=\; \mathbb{E}_{q(z)}\!\left[\log \frac{q(z)}{p(z)}\right] \;\geq\; 0 $$

How surprised you would be, on average, if you believed $q$ and reality was $p$.
Zero when they match. **Never negative** - that fact does all the work in §4.
And not symmetric, so it is not a distance.

In [ ]:
def kl_gauss(m1, s1, m2, s2):
    """Closed form for two 1-D Gaussians."""
    return math.log(s2 / s1) + (s1 ** 2 + (m1 - m2) ** 2) / (2 * s2 ** 2) - 0.5

def kl_monte_carlo(m1, s1, m2, s2, K=400000, seed=0):
    g = np.random.default_rng(seed)
    z = g.normal(m1, s1, K)
    lq = -0.5 * ((z - m1) / s1) ** 2 - math.log(s1 * math.sqrt(2 * math.pi))
    lp = -0.5 * ((z - m2) / s2) ** 2 - math.log(s2 * math.sqrt(2 * math.pi))
    return float((lq - lp).mean())

print('%-34s %10s %12s' % ('', 'closed form', 'Monte Carlo'))
for (m1, s1), (m2, s2) in [((0, 1), (0, 1)), ((0.8, 1), (0, 1)),
                           ((0, 0.4), (0, 1)), ((2.0, 1.4), (0, 1))]:
    print('KL( N(%4.1f,%3.1f) || N(%3.1f,%3.1f) )  %10.4f %12.4f'
          % (m1, s1, m2, s2, kl_gauss(m1, s1, m2, s2), kl_monte_carlo(m1, s1, m2, s2)))

print('\nAnd the asymmetry:')
print('   KL( narrow || wide ) = %.3f' % kl_gauss(0, 0.4, 0, 1))
print('   KL( wide || narrow ) = %.3f   <-- same two distributions' % kl_gauss(0, 1, 0, 0.4))

The closed form agrees with the sampled estimate, so we can trust it - and
because it is closed-form, the KL term in the VAE loss costs nothing and adds
no noise. For a diagonal Gaussian against a standard normal:

$$ D_{\mathrm{KL}}\!\left(\mathcal{N}(\mu,\sigma^2)\,\|\,\mathcal{N}(0,1)\right) = -\frac{1}{2}\sum_{j=1}^{d}\left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right) $$

That is one line of PyTorch, and it is a large part of why VAEs use Gaussians
everywhere. A convenience, not a law of nature.

In [ ]:
def kl_standard_normal(mu, logvar):
    return -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(1)

# check the vector version against the scalar one
m = torch.tensor([[0.8, 0.0, 2.0]]); lv = torch.tensor([[0.0, 2 * math.log(0.4), 2 * math.log(1.4)]])
print('vectorised  %.4f' % kl_standard_normal(m, lv).item())
print('by hand     %.4f' % sum(kl_gauss(a, b, 0, 1) for a, b in [(0.8, 1.0), (0.0, 0.4), (2.0, 1.4)]))

---
## 2. Why we cannot just do the integral

The model is a story: draw $z$ from a simple prior, push it through a decoder,
get an image.

$$ p_\theta(x) \;=\; \int p_\theta(x \mid z)\, p(z)\, dz $$

Maximise that over the training set and you have fitted the model. It is just
maximum likelihood, the same principle as Project 2. **And you cannot compute
it.**

The obvious fallback is §1's lesson: estimate the integral by sampling $z$ from
the prior. Let us try that and see how it goes. (We borrow a trained decoder to
make the test concrete; §5 builds it properly.)

In [ ]:
class VAE(nn.Module):
    def __init__(self, d_z=8, d_hid=256):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(784, d_hid), nn.ReLU(),
                                  nn.Linear(d_hid, d_hid), nn.ReLU())
        self.mu = nn.Linear(d_hid, d_z)           # head 1: the centre of the region
        self.logvar = nn.Linear(d_hid, d_z)       # head 2: its LOG VARIANCE
        self.dec = nn.Sequential(nn.Linear(d_z, d_hid), nn.ReLU(),
                                 nn.Linear(d_hid, d_hid), nn.ReLU(),
                                 nn.Linear(d_hid, 784))
        self.d_z = d_z

    def encode(self, x):
        h = self.body(x); return self.mu(h), self.logvar(h)

def reparameterise(mu, logvar):
    return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

def train_vae(d_z=8, epochs=15, seed=0, beta=1.0, log=False):
    torch.manual_seed(seed)
    m = VAE(d_z).to(DEV)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    hist = {'recon': [], 'kl': []}
    for ep in range(epochs):
        perm = torch.randperm(len(X), generator=g).to(DEV)
        sr = sk = 0.0
        for i in range(0, len(X), 128):
            xb = X[perm[i:i + 128]]
            mu, lv = m.encode(xb)
            z = reparameterise(mu, lv)
            recon = F.binary_cross_entropy_with_logits(m.dec(z), xb, reduction='none').sum(1)
            kl = kl_standard_normal(mu, lv)
            (recon + beta * kl).mean().backward()
            opt.step(); opt.zero_grad()
            sr += recon.sum().item(); sk += kl.sum().item()
        hist['recon'].append(sr / len(X)); hist['kl'].append(sk / len(X))
        if log and (ep % 4 == 0 or ep == epochs - 1):
            print('   epoch %2d   reconstruction %7.2f   KL %6.2f nats'
                  % (ep, hist['recon'][-1], hist['kl'][-1]))
    m.eval(); return m, hist

print('training a reference model (8-dimensional latent)')
t0 = time.time()
ref, _ = train_vae(8, epochs=15)
print('   done in %.0f s' % (time.time() - t0))

In [ ]:
x0 = Xt[:1]
K = 20000
g = torch.Generator().manual_seed(1)
with torch.no_grad():
    # (a) sample z from the PRIOR -- what the integral literally asks for
    z_prior = torch.randn(K, ref.d_z, generator=g).to(DEV)
    lp_prior = -F.binary_cross_entropy_with_logits(
        ref.dec(z_prior), x0.expand(K, 784), reduction='none').sum(1)
    # (b) sample z from the ENCODER -- which knows where to look
    mu0, lv0 = ref.encode(x0)
    z_enc = mu0 + torch.exp(0.5 * lv0) * torch.randn(K, ref.d_z, generator=g).to(DEV)
    lp_enc = -F.binary_cross_entropy_with_logits(
        ref.dec(z_enc), x0.expand(K, 784), reduction='none').sum(1)

thresh = float(lp_enc.median())
print('for ONE fixed test image, in an %d-dimensional latent space:' % ref.d_z)
print('   best  log p(x|z) over %d PRIOR   draws  %8.1f' % (K, lp_prior.max()))
print('   best  log p(x|z) over %d ENCODER draws  %8.1f' % (K, lp_enc.max()))
print('   prior draws at least as good as a TYPICAL encoder draw: %d of %d'
      % (int((lp_prior > thresh).sum()), K))

fig, ax = plt.subplots(figsize=(6.6, 2.8))
bins = np.linspace(float(min(lp_prior.min(), lp_enc.min())), float(lp_enc.max()) + 10, 70)
ax.hist(lp_prior.cpu().numpy(), bins=bins, color='#808080', alpha=.75, label='z from the prior')
ax.hist(lp_enc.cpu().numpy(), bins=bins, color='#1F3864', alpha=.8, label='z from the encoder')
ax.set_yscale('log'); ax.set_xlabel(r'$\log p(x \mid z)$ for one fixed image')
ax.legend(fontsize=8.5); plt.tight_layout(); plt.show()
print('\nAlmost every code is irrelevant to almost every image -- and this is only')
print('%d dimensions. We need something that knows where to look.' % ref.d_z)

### The circle, and how to cut it

The thing that knows where to look is the posterior $p_\theta(z\mid x)$. By Bayes
it needs $p_\theta(x)$ in the denominator, which is the thing we could not
compute. We are going in circles.

**So approximate it.** Train a second network $q_\phi(z\mid x)$ - the encoder -
to guess the posterior directly. One network shared across every image, so
inference for a new image is a single forward pass. That sharing is called
**amortised inference**, and it is why a VAE is fast at test time where
classical variational inference is not.

The word *variational* refers to exactly this move: replace an intractable
quantity with the best member of a family of tractable ones.

---
## 3. The encoder outputs a region

Two heads of size $d_z$: one for $\mu$, one for $\log \sigma^2$.

**Why the log variance?** A linear layer's output can be any real number, and
$\sigma$ must be positive. Parameterising $\log\sigma^2$ makes every output
legal, and $\sigma = \exp(\tfrac12\log\sigma^2)$ is positive by construction.

In [ ]:
probe = VAE(d_z=2).to(DEV)
mu, lv = probe.encode(Xt[:4])
print('mu      ', mu.detach().cpu().numpy().round(3))
print('sigma   ', torch.exp(0.5 * lv).detach().cpu().numpy().round(3))
print('\nAt initialisation sigma is near 1 -- the encoder starts out saying')
print('"I have no idea where this image lives", which IS the prior.')

### The obstacle: you cannot backpropagate through a dice roll

We need the gradient of an expectation over $z \sim q_\phi(z\mid x)$, and $\phi$
controls the distribution we are sampling *from*. There is no derivative of
"draw a random number" with respect to the parameters of the thing you drew it
from.

### The trick: move the randomness to an input

$$ z = \mu_\phi(x) + \sigma_\phi(x) \odot \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I) $$

Now $\varepsilon$ comes from a fixed distribution that depends on nothing, and
$z$ is an ordinary differentiable expression. Same samples, same distribution,
and a gradient that flows.

In [ ]:
mu_t     = torch.zeros(1, 2, requires_grad=True)
logvar_t = torch.zeros(1, 2, requires_grad=True)
z = reparameterise(mu_t, logvar_t)
z.sum().backward()
print('dz/d mu      ', mu_t.grad.numpy(), '  <- exactly 1, since z = mu + ...')
print('dz/d logvar  ', logvar_t.grad.numpy(), '  <- 0.5*sigma*eps, depends on the draw')
print('\nThe gradient survives the sampling step. That is the whole trick.')

In [ ]:
# What is it worth? Compare against the score-function (REINFORCE) estimator,
# which is unbiased and needs no trick. Same loss, same batch, 60 repeats.
def probe_pair(seed):
    torch.manual_seed(seed)
    enc = nn.Sequential(nn.Linear(784, 256), nn.ReLU(),
                        nn.Linear(256, 256), nn.ReLU()).to(DEV)
    hmu, hlv = nn.Linear(256, 8).to(DEV), nn.Linear(256, 8).to(DEV)
    dec = nn.Sequential(nn.Linear(8, 256), nn.ReLU(), nn.Linear(256, 784)).to(DEV)
    return enc, hmu, hlv, dec

def variance_ratios(seed, xb, reps=60):
    enc, hmu, hlv, dec = probe_pair(seed)
    mods = [enc, hmu, hlv, dec]
    def loss_of(z, mu, lv):
        r = F.binary_cross_entropy_with_logits(dec(z), xb, reduction='none').sum(1)
        return r + kl_standard_normal(mu, lv)
    def g_reparam():
        for m_ in mods: m_.zero_grad()
        h = enc(xb); mu, lv = hmu(h), hlv(h)
        loss_of(reparameterise(mu, lv), mu, lv).mean().backward()
        return hmu.weight.grad.flatten().clone()
    def g_score(baseline):
        for m_ in mods: m_.zero_grad()
        h = enc(xb); mu, lv = hmu(h), hlv(h)
        std = torch.exp(0.5 * lv)
        z = (mu + std * torch.randn_like(std)).detach()        # path severed
        logq = (-0.5 * ((z - mu) / std) ** 2 - torch.log(std)).sum(1)
        with torch.no_grad():
            f = loss_of(z, mu, lv)
            if baseline: f = f - f.mean()
        (logq * f).mean().backward()
        return hmu.weight.grad.flatten().clone()
    v = [float(torch.stack([fn() for _ in range(reps)]).var(0).mean())
         for fn in (g_reparam, lambda: g_score(True), lambda: g_score(False))]
    return v[1] / v[0], v[2] / v[0]

xb = X[:256]
rb, rn = zip(*[variance_ratios(s, xb) for s in range(3)])
print('gradient variance, relative to the reparameterised estimator (3 seeds)')
print('   score function + control variate   %6.0fx   (range %.0f-%.0f)'
      % (np.median(rb), min(rb), max(rb)))
print('   score function, no control variate %6.0fx   (range %.3g-%.3g)'
      % (np.median(rn), min(rn), max(rn)))
print('\nSame target, same batch. Quote the order of magnitude, not the digits --')
print('it depends on the architecture and the loss scale.')

---
## 4. The ELBO, and a numerical check that it really is a bound

The derivation from the lecture, in four steps:

$$
\begin{aligned}
\log p_\theta(x) &= \mathbb{E}_{q_\phi(z|x)}\!\left[\log p_\theta(x)\right] && \text{(a constant's average is itself)}\\[2pt]
&= \mathbb{E}_{q}\!\left[\log \frac{p_\theta(x,z)}{p_\theta(z\mid x)}\right] && \text{(Bayes, rearranged)}\\[2pt]
&= \mathbb{E}_{q}\!\left[\log\left(\frac{p_\theta(x,z)}{q_\phi(z\mid x)}\cdot\frac{q_\phi(z\mid x)}{p_\theta(z\mid x)}\right)\right] && \text{(multiply by one)}\\[2pt]
&= \underbrace{\mathbb{E}_{q}\!\left[\log\frac{p_\theta(x,z)}{q_\phi(z\mid x)}\right]}_{\text{the ELBO } \mathcal{L}} \;+\; \underbrace{D_{\mathrm{KL}}\!\left(q_\phi(z|x)\,\|\,p_\theta(z|x)\right)}_{\geq\,0}
\end{aligned}
$$

A KL is never negative, so $\mathcal{L} \leq \log p_\theta(x)$. And the gap is
*exactly* how wrong the encoder is: the bound is tight when
$q_\phi(z\mid x) = p_\theta(z\mid x)$.

Rearranged into the form you code:

$$ \mathcal{L} \;=\; \underbrace{\mathbb{E}_{q}\!\left[\log p_\theta(x\mid z)\right]}_{\text{rebuild the image}} \;-\; \underbrace{D_{\mathrm{KL}}\!\left(q_\phi(z|x)\,\|\,p(z)\right)}_{\text{stay near the prior}} $$

### Checking the bound without knowing the true posterior

We cannot compute $\log p_\theta(x)$ directly - that was the whole problem. But
we can get a *tighter and tighter* estimate of it with the importance-weighted
bound (Burda, Grosse & Salakhutdinov, ICLR 2016):

$$ \log p_\theta(x) \;\approx\; \log \frac{1}{K}\sum_{k=1}^{K}\frac{p_\theta(x\mid z_k)\,p(z_k)}{q_\phi(z_k\mid x)}, \qquad z_k \sim q_\phi(z\mid x) $$

$K=1$ gives back the ELBO exactly, and larger $K$ is provably tighter. So if the
theory is right, this number should **rise with $K$ and never fall**.

In [ ]:
def iw_bound(m, xs, K, seed=0, bs=50):
    """K=1 recovers the ELBO. Larger K is a tighter lower bound on log p(x)."""
    out = []
    g = torch.Generator().manual_seed(seed)
    with torch.no_grad():
        for i in range(0, len(xs), bs):
            x = xs[i:i + bs]
            mu, lv = m.encode(x)
            lw = []
            for _ in range(K):
                eps = torch.randn(mu.shape, generator=g).to(DEV)
                z = mu + torch.exp(0.5 * lv) * eps
                log_pxz = -F.binary_cross_entropy_with_logits(
                    m.dec(z), x, reduction='none').sum(1)
                log_pz  = (-0.5 * z ** 2 - 0.5 * math.log(2 * math.pi)).sum(1)
                log_qzx = (-0.5 * eps ** 2 - 0.5 * math.log(2 * math.pi) - 0.5 * lv).sum(1)
                lw.append(log_pxz + log_pz - log_qzx)
            lw = torch.stack(lw)
            out.append(torch.logsumexp(lw, 0) - math.log(K))
    return float(torch.cat(out).mean())

xs = Xt[:500]
print('%6s %22s' % ('K', 'estimate of log p(x)'))
vals = []
for K in (1, 2, 5, 20, 100):
    v = iw_bound(ref, xs, K); vals.append(v)
    print('%6d %22.3f%s' % (K, v, '   <- this is exactly the ELBO' if K == 1 else ''))
print('\nmonotone increasing:', all(b >= a - 1e-6 for a, b in zip(vals, vals[1:])))
print('\nThe ELBO sits %.2f nats below the K=100 estimate. That gap is the price' % (vals[-1] - vals[0]))
print('of our encoder not being the true posterior -- exactly the KL term in the')
print('derivation above. Nothing else could have produced it.')

That is the bound, demonstrated. The ELBO is the $K=1$ row; every tighter
estimate lies above it, and the gap is the encoder's error made visible.

**Practical consequence.** If you want to *report* a log-likelihood, use the
importance-weighted estimate with $K$ in the hundreds. If you only need a
*ranking* - as in anomaly detection next lecture - the plain ELBO is usually
fine, and much cheaper.

---
## 5. Train it, and watch the two terms

**Always log the reconstruction and KL terms separately.** A single total hides
what the model is doing. The pattern to look for: reconstruction falls steeply
while the KL *rises*. That rise is the model deciding the latent code is worth
paying for - it is buying information about $x$ at a cost in KL nats.

In [ ]:
print('training a 2-dimensional VAE (small enough to draw the whole latent space)')
t0 = time.time()
vae2, hist = train_vae(2, epochs=20, log=True)
print('   done in %.0f s' % (time.time() - t0))

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ep = range(1, len(hist['recon']) + 1)
ax.plot(ep, hist['recon'], 'o-', color='#1F3864', lw=2, ms=4)
ax.set_xlabel('epoch'); ax.set_ylabel('reconstruction, nats', color='#1F3864')
ax.tick_params(axis='y', labelcolor='#1F3864')
ax2 = ax.twinx()                 # the KL is tiny next to recon, so it needs its own axis
ax2.plot(ep, hist['kl'], 's-', color='#9C2B2B', lw=2, ms=4)
ax2.set_ylabel('KL, nats', color='#9C2B2B'); ax2.tick_params(axis='y', labelcolor='#9C2B2B')
ax.set_title('the two terms pull in opposite directions')
plt.tight_layout(); plt.show()
print('reconstruction %.1f -> %.1f nats' % (hist['recon'][0], hist['recon'][-1]))
print('KL             %.1f -> %.1f nats   <-- it RISES' % (hist['kl'][0], hist['kl'][-1]))

In [ ]:
# the latent space, and the decoder as a chart
with torch.no_grad():
    MU, _ = vae2.encode(Xt)
Z = MU.cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.8))
sc = ax[0].scatter(Z[:, 0], Z[:, 1], c=Yt.numpy(), s=3, cmap='tab10', alpha=.6)
ax[0].set_title('the codes now sit in a tidy blob of unit scale\n(range %.1f to %.1f)'
                % (Z.min(), Z.max()), fontsize=10)
ax[0].set_xlabel('$z_1$'); ax[0].set_ylabel('$z_2$')
plt.colorbar(sc, ax=ax[0], ticks=range(10), label='digit')

R, lim = 18, 2.5
gy, gx = np.meshgrid(np.linspace(lim, -lim, R), np.linspace(-lim, lim, R))
G = torch.tensor(np.stack([gx.ravel(), gy.ravel()], 1), dtype=torch.float32, device=DEV)
with torch.no_grad():
    imgs = torch.sigmoid(vae2.dec(G)).reshape(R, R, 28, 28).cpu().numpy()
ax[1].imshow(imgs.transpose(0, 2, 1, 3).reshape(R * 28, R * 28), cmap='gray',
             extent=[-lim, lim, -lim, lim])
ax[1].set_title('decode a regular grid: digits everywhere, changing smoothly', fontsize=10)
ax[1].set_xlabel('$z_1$'); ax[1].set_ylabel('$z_2$')
plt.tight_layout(); plt.show()

In [ ]:
# the experiment that failed last lecture, repeated
g = torch.Generator().manual_seed(7)
with torch.no_grad():
    show(torch.sigmoid(vae2.dec(torch.randn(10, 2, generator=g).to(DEV))),
         title='z ~ N(0, I), decoded -- the SAME naive scheme that failed on the autoencoder')
    i3 = int((Yt == 3).nonzero()[0]); i8 = int((Yt == 8).nonzero()[0])
    t = torch.linspace(0, 1, 10, device=DEV).unsqueeze(1)
    m1, _ = vae2.encode(Xt[i3:i3+1]); m2, _ = vae2.encode(Xt[i8:i8+1])
    show(torch.sigmoid(vae2.dec((1 - t) * m1 + t * m2)), title='interpolating: a 3 to an 8')
print('Every intermediate image is a plausible digit, and the naive sampling')
print('scheme works. The difference is not the architecture -- it is that we')
print('TRAINED the space to make these sensible things to do.')
print('\nThey are also blurry. That is next lecture, and it is not a bug you can')
print('tune away: it follows from the likelihood we chose.')

---
## 6. The regression test every VAE should pass

Strip every non-linearity out - linear encoder heads, linear decoder, Gaussian
likelihood - and a VAE becomes probabilistic PCA. **If your implementation
cannot reproduce PCA with the non-linearities deleted, the bug is in your ELBO,
not your architecture.**

We compare subspaces with *principal angles*: 0° means identical, 90° means
orthogonal.

In [ ]:
class LinearVAE(nn.Module):
    def __init__(self, d_z=2):
        super().__init__()
        self.mu = nn.Linear(784, d_z); self.logvar = nn.Linear(784, d_z)
        self.dec = nn.Linear(d_z, 784)          # no non-linearity anywhere

    def encode(self, x):
        return self.mu(x), self.logvar(x)

mean_x = X.mean(0, keepdim=True)
Xc = X - mean_x
_, _, Vt = torch.linalg.svd(Xc, full_matrices=False)
V = Vt.T

torch.manual_seed(0)
lin = LinearVAE(2).to(DEV)
opt = torch.optim.Adam(lin.parameters(), lr=3e-3)
g = torch.Generator().manual_seed(0)
for ep in range(40):
    perm = torch.randperm(len(Xc), generator=g).to(DEV)
    for i in range(0, len(Xc), 256):
        xb = Xc[perm[i:i + 256]]
        mu, lv = lin.encode(xb)
        z = reparameterise(mu, lv)
        recon = 0.5 * ((lin.dec(z) - xb) ** 2).sum(1)      # Gaussian, sigma^2 = 1
        (recon + kl_standard_normal(mu, lv)).mean().backward()
        opt.step(); opt.zero_grad()
lin.eval()

Wd = lin.dec.weight.detach()
Q, _ = torch.linalg.qr(Wd)                     # orthonormal basis of the decoder's columns
s = torch.linalg.svdvals(Q.T @ V[:, :2]).clamp(-1, 1)
ang = torch.rad2deg(torch.arccos(s))
print('principal angles between the linear VAE and PCA: %.2f and %.2f degrees'
      % (ang[0], ang[1]))
print('\nNear zero: the same subspace. Not exactly zero -- finite SGD, and the')
print('fixed sigma^2 = 1 is not the maximum-likelihood value.')

fig, ax = plt.subplots(2, 2, figsize=(5, 5.2))
for i in range(2):
    ax[0, i].imshow(V[:, i].cpu().reshape(28, 28), cmap='RdBu_r')
    ax[0, i].set_title('PCA axis %d' % (i + 1), fontsize=10); ax[0, i].axis('off')
    w = Wd[:, i].cpu(); w = w * torch.sign((w * V[:, i].cpu()).sum())     # fix the sign
    ax[1, i].imshow(w.reshape(28, 28), cmap='RdBu_r')
    ax[1, i].set_title('linear VAE column %d' % (i + 1), fontsize=10); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

---
## 7. Summary, and what to carry forward

| piece | what it is | what breaks without it |
|---|---|---|
| **encoder** $q_\phi(z\mid x)$ | a network with $\mu$ and $\log\sigma^2$ heads | you would have to optimise a $z$ per data point |
| **reparameterisation** | $z = \mu + \sigma\odot\varepsilon$ | no gradient through the sample, or a ~10⁵-fold variance increase |
| **decoder** $p_\theta(x\mid z)$ | a network returning *distribution parameters* | no likelihood, no principled loss |
| **the KL term** | $D_{\mathrm{KL}}(q_\phi(z|x)\,\|\,p(z))$ | last lecture's autoencoder: holes, no scale, nothing to sample |

### A checklist

1. **Log the two terms separately.** A total that looks fine can hide a KL that
   has gone to zero, which means your latent space is empty.
2. **The ELBO is a bound, not an estimate.** If you want to report a
   log-likelihood, use the importance-weighted version from §4.
3. **Delete the non-linearities and check you get PCA.** Ten lines, and it
   catches real bugs.
4. **Name your decoder distribution out loud.** We used Bernoulli here, which is
   why the loss is `binary_cross_entropy_with_logits` and why pixels had to stay
   in $[0,1]$. Next lecture shows what goes wrong when you forget this.

### Exercises

1. Set the KL weight to 0 in `train_vae` (pass `beta=0`). You now have a plain
   autoencoder. Reconstructions should improve. Sample from $\mathcal{N}(0,I)$
   and decode - what do you get, and why?
2. Set `beta=16` at `d_z=10` and count how many latent dimensions still carry
   information (look at `mu.var(0)`). This is **posterior collapse**.
3. In §4, plot the IW estimate against $K$ on a log axis. Does it look like it
   is converging, and to what?
4. Replace the Bernoulli decoder with a Gaussian one using summed squared error
   and *no* $1/\sigma^2$. Retrain. What happens to the latent space, and why?
   (Hint: what $\beta$ did you just choose without noticing?)
5. Anneal the KL weight from 0 to 1 over the first 5 epochs - "KL warm-up",
   Bowman et al. 2016. Does it change the final number of active dimensions?

### References

- Kingma & Welling. Auto-Encoding Variational Bayes. *ICLR* 2014.
  arXiv:1312.6114.
- Rezende, Mohamed & Wierstra. Stochastic Backpropagation and Approximate
  Inference in Deep Generative Models. *ICML* 2014.
- Kingma & Welling. An Introduction to Variational Autoencoders. *Foundations
  and Trends in Machine Learning*, 2019. arXiv:1906.02691. §2.2 is the
  derivation we used, by the same route.
- Burda, Grosse & Salakhutdinov. Importance Weighted Autoencoders. *ICLR* 2016.
- Blei, Kucukelbir & McAuliffe. Variational Inference: A Review for
  Statisticians. *JASA* 112(518):859-877, 2017.
- Doersch. Tutorial on Variational Autoencoders. arXiv:1606.05908, 2016.
- Lucas, Tucker, Grosse & Norouzi. Don't Blame the ELBO! A Linear VAE
  Perspective on Posterior Collapse. *NeurIPS* 2019.
- Bowman et al. Generating Sentences from a Continuous Space. *CoNLL* 2016.